In [ ]:
import os
from datetime import date, datetime, timedelta

import pandas as pd
import requests

from sqlalchemy import create_engine, text


# Параметри підключення до PostgreSQL
DB_USER = "postgres"
DB_PASSWORD = "postgres123"
DB_HOST = "127.0.0.1"
DB_PORT = "5432"
DB_NAME = "nbu_rates"


# Створення підключення
CONNECTION_URL = (
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}"
    f"@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

engine = create_engine(CONNECTION_URL)


# Перевірка підключення
try:
    with engine.connect() as connection:

        print("Підключення до PostgreSQL успішне!")

        # Отримання назв таблиць схеми public
        tables = pd.read_sql(
            text("""
                SELECT table_name
                FROM information_schema.tables
                WHERE table_schema = 'public'
                  AND table_type = 'BASE TABLE'
                ORDER BY table_name
            """),
            connection
        )

        print("\nТаблиці схеми public:")
        display(tables)

except Exception as error:
    print("Помилка підключення до PostgreSQL:")
    print(error)



Підключення до PostgreSQL успішне!

Таблиці схеми public:


,table_name
0,current_rates
1,load_watermarks
2,rates_history


In [45]:
NBU_CURRENT_URL = "https://bank.gov.ua/NBUStatService/v1/statdirectory/exchange?json"
NBU_DATE_URL = "https://bank.gov.ua/NBUStatService/v1/statdirectory/exchange"

API_TIMEOUT = 10

def fetch_nbu_rates(url, params=None):
    """Отримати та перевірити відповідь API до будь-якого запису в БД."""
    response = requests.get(url, params=params, timeout=API_TIMEOUT)
    response.raise_for_status()

    payload = response.json()
    if not isinstance(payload, list):
        raise ValueError("API повернув дані не у форматі списку JSON.")
    if not payload:
        raise ValueError("API повернув порожній список курсів.")

    required = {"r030", "txt", "rate", "cc", "exchangedate"}
    for item in payload:
        if not isinstance(item, dict) or not required.issubset(item):
            raise ValueError("Відповідь API не містить усіх обов'язкових полів.")

    return payload


def prepare_current_rates(payload):
    df = pd.DataFrame(payload).rename(columns={
        "cc": "currency_code",
        "r030": "currency_code_num",
        "txt": "currency_name",
        "exchangedate": "rate_date",
    })
    df["rate_date"] = pd.to_datetime(df["rate_date"], format="%d.%m.%Y").dt.date
    df = df[[
        "currency_code",
        "currency_code_num",
        "currency_name",
        "rate",
        "rate_date",
    ]].copy()
    return df


def full_load():
    payload = fetch_nbu_rates(NBU_CURRENT_URL)
    df = prepare_current_rates(payload)

    # Очищення та вставка атомарні; структура таблиці не змінюється.
    with engine.begin() as connection:
        connection.execute(text("DELETE FROM current_rates"))
        df.to_sql(
            "current_rates",
            con=connection,
            schema="public",
            if_exists="append",
            index=False,
            method="multi",
        )

    print(f"Повне завантаження завершено. Завантажено рядків: {len(df)}")
    return len(df)


In [46]:
# Два послідовні виклики мають залишити однакову кількість рядків.
count_after_first = full_load()
with engine.connect() as connection:
    count_current_1 = connection.execute(
        text("SELECT COUNT(*) FROM current_rates")
    ).scalar_one()
print("Кількість рядків після першого виклику:", count_current_1)

count_after_second = full_load()
with engine.connect() as connection:
    count_current_2 = connection.execute(
        text("SELECT COUNT(*) FROM current_rates")
    ).scalar_one()
print("Кількість рядків після другого виклику:", count_current_2)

with engine.connect() as connection:
    display(pd.read_sql(
        text("""
            SELECT currency_code, currency_code_num, currency_name, rate, rate_date
            FROM current_rates
            ORDER BY currency_code
            LIMIT 5
        """),
        connection
    ))


Повне завантаження завершено. Завантажено рядків: 45
Кількість рядків після першого виклику: 45
Повне завантаження завершено. Завантажено рядків: 45
Кількість рядків після другого виклику: 45


,currency_code,currency_code_num,currency_name,rate,rate_date
0,AED,784,Дирхам ОАЕ,12.39720,2026-11-03
1,AUD,36,Австралійський долар,31.90670,2026-11-03
2,AZN,944,Азербайджанський манат,26.78910,2026-11-03
3,BDT,50,Така,0.36598,2026-11-03
4,CAD,124,Канадський долар,32.58400,2026-11-03


In [47]:
def load_rates_for_date(batch_date):
    """
    Завантажити курси за одну дату.
    batch_date: текст у форматі YYYY-MM-DD.
    Повертає кількість фактично доданих рядків.
    """
    parsed_date = datetime.strptime(batch_date, "%Y-%m-%d").date()
    api_date = parsed_date.strftime("%Y%m%d")

    payload = fetch_nbu_rates(
        NBU_DATE_URL,
        params={"date": api_date, "json": ""},
    )

    df = pd.DataFrame(payload).rename(columns={
        "cc": "currency_code",
        "txt": "currency_name",
        "exchangedate": "rate_date",
    })
    df["rate_date"] = pd.to_datetime(
        df["rate_date"], format="%d.%m.%Y"
    ).dt.date
    df = df[["currency_code", "currency_name", "rate", "rate_date"]].copy()

    # Звіряємо ключі в БД; передаємо дату параметром SQL-запиту.
    with engine.connect() as connection:
        existing = pd.read_sql(
            text("""
                SELECT currency_code
                FROM rates_history
                WHERE rate_date = :batch_date
            """),
            connection,
            params={"batch_date": parsed_date},
        )

    existing_codes = set(existing["currency_code"].tolist())
    new_rows = df[~df["currency_code"].isin(existing_codes)].copy()
    added_count = len(new_rows)
    skipped_count = len(df) - added_count

    if added_count:
        with engine.begin() as connection:
            new_rows.to_sql(
                "rates_history",
                con=connection,
                schema="public",
                if_exists="append",
                index=False,
                method="multi",
            )

    print(
        f"{parsed_date}: додано {added_count} рядків, "
        f"пропущено {skipped_count} рядків."
    )
    return added_count


In [48]:
# Дати за умовою не змінювати.
added_2026_08_03 = load_rates_for_date("2026-08-03")
added_2026_08_04 = load_rates_for_date("2026-08-04")
added_2026_08_03_repeat = load_rates_for_date("2026-08-03")

with engine.connect() as connection:
    display(pd.read_sql(
        text("""
            SELECT rate_date, COUNT(*) AS rows_count
            FROM rates_history
            GROUP BY rate_date
            ORDER BY rate_date
        """),
        connection
    ))


2026-08-03: додано 0 рядків, пропущено 45 рядків.
2026-08-04: додано 0 рядків, пропущено 45 рядків.
2026-08-03: додано 0 рядків, пропущено 45 рядків.


,rate_date,rows_count
0,2026-08-03,45
1,2026-08-04,45
2,2026-08-05,45
3,2026-08-06,45
4,2026-08-07,45


In [50]:
# Початковий водяний знак додається параметризованим SQL-запитом.
with engine.begin() as connection:
    connection.execute(
        text("""
            INSERT INTO load_watermarks (pipeline_name, last_loaded_date)
            VALUES (:pipeline_name, :last_loaded_date)
            ON CONFLICT (pipeline_name) DO UPDATE
            SET last_loaded_date = EXCLUDED.last_loaded_date
        """),
        {
            "pipeline_name": "nbu_rates",
            "last_loaded_date": date(2026, 8, 4),
        },
    )

print("Початковий водяний знак встановлено.")


Початковий водяний знак встановлено.


In [51]:
def incremental_load(until_date):
    """
    Завантажити всі дати після поточного watermark до until_date включно.
    API викликається лише через load_rates_for_date.
    Повертає сумарну кількість доданих рядків.
    """
    end_date = datetime.strptime(until_date, "%Y-%m-%d").date()

    with engine.connect() as connection:
        last_loaded_date = connection.execute(
            text("""
                SELECT last_loaded_date
                FROM load_watermarks
                WHERE pipeline_name = :pipeline_name
            """),
            {"pipeline_name": "nbu_rates"},
        ).scalar_one()

    dates_to_load = []
    next_date = last_loaded_date + timedelta(days=1)
    while next_date <= end_date:
        dates_to_load.append(next_date)
        next_date += timedelta(days=1)

    total_added = 0
    for load_date in dates_to_load:
        added = load_rates_for_date(load_date.isoformat())
        total_added += added

        # Рухаємо watermark лише після успішного завершення завантаження дати.
        with engine.begin() as connection:
            connection.execute(
                text("""
                    UPDATE load_watermarks
                    SET last_loaded_date = :loaded_date
                    WHERE pipeline_name = :pipeline_name
                """),
                {
                    "loaded_date": load_date,
                    "pipeline_name": "nbu_rates",
                },
            )

    return total_added


In [52]:
# Перший запуск: завантаження 2026-08-05, 2026-08-06 та 2026-08-07.
incremental_result_1 = incremental_load("2026-08-07")
print("Додано рядків під час першого incremental_load:", incremental_result_1)

with engine.connect() as connection:
    display(pd.read_sql(
        text("SELECT pipeline_name, last_loaded_date FROM load_watermarks"),
        connection
    ))


2026-08-05: додано 0 рядків, пропущено 45 рядків.
2026-08-06: додано 0 рядків, пропущено 45 рядків.
2026-08-07: додано 0 рядків, пропущено 45 рядків.
Додано рядків під час першого incremental_load: 0


,pipeline_name,last_loaded_date
0,nbu_rates,2026-08-07


In [53]:
# Другий запуск: watermark уже дорівнює 2026-08-07,
# тому список дат порожній, API не викликається.
incremental_result_2 = incremental_load("2026-08-07")
print("Додано рядків під час другого incremental_load:", incremental_result_2)

with engine.connect() as connection:
    display(pd.read_sql(
        text("""
            SELECT rate_date, COUNT(*) AS rows_count
            FROM rates_history
            GROUP BY rate_date
            ORDER BY rate_date
        """),
        connection
    ))
    display(pd.read_sql(
        text("SELECT pipeline_name, last_loaded_date FROM load_watermarks"),
        connection
    ))


Додано рядків під час другого incremental_load: 0


,rate_date,rows_count
0,2026-08-03,45
1,2026-08-04,45
2,2026-08-05,45
3,2026-08-06,45
4,2026-08-07,45


,pipeline_name,last_loaded_date
0,nbu_rates,2026-08-07
